# Tool Calls 示例

演示如何使用 OpenAI Function Calling（工具调用）让模型调用外部函数获取星座运势。

In [1]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
base_url = os.getenv('OPENAI_API_BASE')

# 初始化 OpenAI 客户端
client = OpenAI(
    base_url=base_url,
    api_key=api_key
)

## 1. 为模型定义可调用工具列表

In [2]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_horoscope",
            "description": "获取指定星座的今日运势",
            "parameters": {
                "type": "object",
                "properties": {
                    "sign": {
                        "type": "string",
                        "description": "星座名称，如金牛座或水瓶座",
                    },
                },
                "required": ["sign"],
            },
        }
    },
]

## 2. 使用定义的工具提示模型

In [3]:
# 创建一个消息列表，随着时间推移会不断添加内容
messages = [
    {"role": "user", "content": "我的运势如何？我是水瓶座。"}
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    tools=tools,
    messages=messages,
)

print("模型初始输出:")
print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))

模型初始输出:
{
  "id": "chatcmpl-DXPoSkDn9pQX0GB0CTVHRvvgIoP8e",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": null,
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": [
          {
            "id": "call_vva3VkTqi4U6DjV5eQ9EEiKM",
            "function": {
              "arguments": "{\"sign\":\"水瓶座\"}",
              "name": "get_horoscope"
            },
            "type": "function"
          }
        ]
      }
    }
  ],
  "created": 1776856420,
  "model": "gpt-4o-mini",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": "fp_eb37e061ec",
  "usage": {
    "completion_tokens": 19,
    "prompt_tokens": 71,
    "total_tokens": 90,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
   

## 3. 执行 get_horoscope 函数逻辑

In [4]:
# 保存函数调用输出以供后续请求使用
function_call = None
function_call_arguments = None
messages.append(response.choices[0].message)

# 检查模型是否想要调用函数
if response.choices[0].message.tool_calls:
    tool_call = response.choices[0].message.tool_calls[0]
    function_call = tool_call
    function_call_arguments = json.loads(tool_call.function.arguments)


def get_horoscope(sign):
    return f"{sign}: 下周二你将结识一只小水獭。"


# 执行函数
result = {"horoscope": get_horoscope(function_call_arguments["sign"])}
print("函数执行结果:", result)

函数执行结果: {'horoscope': '水瓶座: 下周二你将结识一只小水獭。'}


## 4. 向模型提供函数调用结果

In [5]:
messages.append({
    "tool_call_id": function_call.id,
    "role": "tool",
    "name": "get_horoscope",
    "content": json.dumps(result),
})

print("消息流程:")
for i, message in enumerate(messages):
    if isinstance(message, dict):
        role = message.get('role', 'unknown')
        if role == 'user':
            print(f"{i + 1}. 用户输入: {message.get('content', '')}")
        elif role == 'tool':
            content = json.loads(message.get('content', '{}'))
            print(f"{i + 1}. 工具返回: {content}")
    else:
        print(f"{i + 1}. 助手: 调用工具 {message.tool_calls[0].function.name if message.tool_calls else '无工具调用'}")

消息流程:
1. 用户输入: 我的运势如何？我是水瓶座。
2. 助手: 调用工具 get_horoscope
3. 工具返回: {'horoscope': '水瓶座: 下周二你将结识一只小水獭。'}


## 5. 模型根据工具结果给出最终响应

In [6]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    tools=tools,
    messages=messages,
)

print("最终输出:")
print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))
print("\n" + response.choices[0].message.content)

最终输出:
{
  "id": "chatcmpl-DXPq38v8uCd1M0CcyAdIxmtUXPrbP",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "今天你的运势是：水瓶座的人下周二你将会结识一位小水瓶。这可能会给你带来一些新的交流和合作机会。保持开放的心态，期待新的可能性！",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null
      }
    }
  ],
  "created": 1776856519,
  "model": "gpt-4o-mini",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": "fp_eb37e061ec",
  "usage": {
    "completion_tokens": 50,
    "prompt_tokens": 170,
    "total_tokens": 220,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audio_tokens": 0,
      "reasoning_tokens": 0,
      "rejected_prediction_tokens": 0
    },
    "prompt_tokens_details": {
      "audio_tokens": 0,
      "cached_tokens": 0
    }
  }
}

今天你的运势是：水瓶座的人下周二你将会结识一位小水瓶。这可能会给你带来一些新的交流和合作机会。保持开